# Constraint Generation

In [ ]:
import os
import re
import json
import time
import pandas as pd
from openai import OpenAI
from tenacity import retry, wait_exponential, stop_after_attempt
from collections import Counter
from itertools import combinations
from tqdm import tqdm

In [ ]:
client = OpenAI(
  api_key="REDACTED_USE_ENVIRONMENT_VARIABLE"
)

In [ ]:
ANNOTATION_MODEL = "o3"
REASONING        = {"effort": "medium"}
SLEEP_SEC        = 1.0
PILOT_N          = None   # set to None for full run

QUESTION_COLS = [f"Q{i}" for i in range(1, 8)]  # question1 … question7

In [ ]:
QUESTION_COLS

In [ ]:
df = pd.read_excel("/content/Puzzle_Dataset.xlsx")

# Confirm expected columns
assert "Puzzle_Number"  in df.columns
assert "Context" in df.columns
assert "Clues"          in df.columns
for col in QUESTION_COLS:
    assert col in df.columns, f"Missing expected column: {col}"

if PILOT_N is not None:
    df = df.iloc[:PILOT_N].copy()

print(f"Puzzles to annotate: {len(df)}")
print(f"Question columns   : {QUESTION_COLS}")
print(f"Questions per puzzle (non-null counts):")
print(df[QUESTION_COLS].notna().sum(axis=1).value_counts().sort_index())

# Constraint Extraction Prompt

In [ ]:
CONSTRAINT_EXTRACTION_SYSTEM = """
You are annotating logic puzzles by extracting structured constraints.

Your goal is to convert natural language clues into formal constraint representations.

-----------------------------------
CANONICAL CONSTRAINT TYPES
-----------------------------------
Use ONLY one of the following types:

- ordering: pos(A) < pos(B)
- adjacency: |pos(A) - pos(B)| = 1
- distance: |pos(A) - pos(B)| = k
- absolute: pos(A) = k
- non_adjacency: |pos(A) - pos(B)| != 1
- all_different: all unique
- conditional: if A then B
- disjunctive: A or B
- counting: exactly / at least / at most k items satisfy a condition
- grouping: A and B belong to same or different group/set
- other: does not fit any type above

-----------------------------------
INSTRUCTIONS
-----------------------------------
1. Break the puzzle into individual constraints (one per clue).
2. Include important implicit constraints (e.g., all_different).
3. For each constraint output:
   - constraint (short description)
   - type
   - variables
   - arity
   - formal

4. Use pos(X) for positions.
5. Be consistent with variable names.

-----------------------------------
OUTPUT FORMAT (STRICT JSON)
-----------------------------------
Return ONLY valid JSON:

{
  "constraints": [
    {
      "constraint": "...",
      "type": "...",
      "variables": [],
      "arity": 0,
      "formal": "..."
    }
  ]
}

-----------------------------------
PUZZLE
-----------------------------------
"""

CONSTRAINT_EXTRACTION_USER_TEMPLATE = """{puzzle_context}

{clues}"""

# Question Type Classification

In [ ]:
QUESTION_TYPE_SYSTEM = """
You are annotating logic puzzle questions.

Classify the QUESTION into exactly one type:

- valid_configuration
- must_be_true
- must_be_false
- complete_solution
- conditional_inference
- optimization
- other

Return ONLY JSON:

{
  "question_type": "...",
  "explanation": "..."
}

-----------------------------------
QUESTION
-----------------------------------
"""

QUESTION_TYPE_USER_TEMPLATE = """{question}"""

In [ ]:
@retry(wait=wait_exponential(min=1, max=16), stop=stop_after_attempt(3))
def call_o3(system: str, user: str, label: str = "") -> str:
    try:
        response = client.responses.create(
            model=ANNOTATION_MODEL,
            reasoning=REASONING,
            input=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ],
        )
        return response.output_text.strip()
    except Exception as e:
        print(f"[{label}] API error: {e}")
        raise


def parse_json_output(raw: str, label: str = "") -> dict:
    cleaned = re.sub(r"```(?:json)?", "", raw).strip().rstrip("`").strip()
    match = re.search(r"\{.*\}", cleaned, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError as e:
            print(f"[{label}] JSON parse error: {e}\nRaw: {raw[:300]}")
    return {"error": "json_parse_failed", "raw": raw[:500]}

In [ ]:
@retry(wait=wait_exponential(min=1, max=16), stop=stop_after_attempt(3))
def call_gpt4o(system: str, user: str, label: str = "") -> str:
    try:
        response = client.chat.completions.create(
            model="gpt-4o",
            temperature=0,
            messages=[
                {"role": "system", "content": system},
                {"role": "user",   "content": user},
            ]
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"[{label}] API error: {e}")
        raise

# Constraint Extraction

In [ ]:
def extract_constraints(row: pd.Series) -> dict:
    user_msg = CONSTRAINT_EXTRACTION_USER_TEMPLATE.format(
        puzzle_context=row["Context"],
        clues=row["Clues"]
    )
    raw = call_o3(
        CONSTRAINT_EXTRACTION_SYSTEM,
        user_msg,
        label=f"CONSTRAINTS P{row['Puzzle_Number']}"
    )
    #raw = call_gpt4o(
    #CONSTRAINT_EXTRACTION_SYSTEM,
    #user_msg,
    #label=f"CONSTRAINTS P{row['Puzzle_Number']}")

    return parse_json_output(raw, label=f"CONSTRAINTS P{row['Puzzle_Number']}")


constraint_results = {}   # Puzzle_Number -> annotation dict

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Constraints"):
    constraint_results[row["Puzzle_Number"]] = extract_constraints(row)
    time.sleep(SLEEP_SEC)

In [ ]:
constraint_results[5]

# Question Type Classification

In [ ]:
def classify_question(puzzle_number, question_col: str, question_text: str) -> dict:
    user_msg = QUESTION_TYPE_USER_TEMPLATE.format(question=question_text)
    raw = call_o3(
        QUESTION_TYPE_SYSTEM,
        user_msg,
        label=f"QTYPE P{puzzle_number} {question_col}"
    )
    #raw = call_gpt4o(
    #QUESTION_TYPE_SYSTEM,
    #user_msg,
    #label=f"QTYPE P{puzzle_number} {question_col}")
    result = parse_json_output(raw, label=f"QTYPE P{puzzle_number} {question_col}")
    # Tag with identifiers so the long-format record is self-contained
    result["Puzzle_Number"]   = puzzle_number
    result["question_col"]    = question_col
    result["question_number"] = int(question_col.replace("Q", ""))
    return result


qtype_records = []   # one record per (puzzle, question) pair

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Question types"):
    for col in QUESTION_COLS:
        if pd.isna(row[col]) or str(row[col]).strip() == "":
            continue
        record = classify_question(row["Puzzle_Number"], col, row[col])
        qtype_records.append(record)
    time.sleep(SLEEP_SEC)

qtype_df = pd.DataFrame(qtype_records)
print(f"\nQuestion type records: {len(qtype_df)}")

In [ ]:
qtype_df.head()

In [ ]:
def constraint_type_list(ann: dict) -> list:
    return [c["type"] for c in ann.get("constraints", [])]

df["constraint_annotation"] = df["Puzzle_Number"].map(
    lambda p: json.dumps(constraint_results.get(p, {}))
)
df["constraint_types"] = df["Puzzle_Number"].map(
    lambda p: constraint_type_list(constraint_results.get(p, {}))
)

In [ ]:
# Puzzle-level: context, clues, constraint annotations
df.to_csv("annotated_puzzles_wide.csv", index=False)

# Question-level: one row per (puzzle, question) with type classification
qtype_df.to_csv("annotated_question_types.csv", index=False)

print(f"Saved {len(df)} puzzle rows  -> annotated_puzzles_wide.csv")
print(f"Saved {len(qtype_df)} question rows -> annotated_question_types.csv")

# Constraint Type Analysis

In [ ]:
df = pd.read_csv("/content/annotated_puzzles_wide_final.csv")
question_types = pd.read_csv("/content/annotated_question_types_final.csv")

In [ ]:
df.head()

In [ ]:
import ast

df['constraint_types'] = df['constraint_types'].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

In [ ]:

# explode list of types into rows
types_exploded = df['constraint_types'].explode()

# distribution
type_counts = types_exploded.value_counts()

print(type_counts)

In [ ]:
df['num_types'] = df['constraint_types'].apply(len)

distribution = df['num_types'].value_counts().sort_index()
print(distribution)

In [ ]:
df['constraint_mode'] = df['num_types'].apply(
    lambda x: 'pure' if x == 1 else 'mixed'
)

mode_dist = df['constraint_mode'].value_counts()
print(mode_dist)

In [ ]:
df['constraint_types'].value_counts().head(20)

In [ ]:
# A. Constraint count per puzzle

df['num_constraints'] = df['constraint_annotation'].apply(len)
df['num_constraints'].describe()

In [ ]:
# B. Type diversity per puzzle (better signal); Even if everything is “mixed”, how mixed?

df['type_diversity'] = df['constraint_types'].apply(lambda x: len(set(x)))
df['type_diversity'].describe()

# "Other" type Analysis

In [ ]:
import ast
import json

def safe_parse(x):
    if isinstance(x, dict):
        return x
    if isinstance(x, str):
        try:
            return json.loads(x)
        except:
            return ast.literal_eval(x)
    return x

In [ ]:
df['constraint_annotation'] = df['constraint_annotation'].apply(safe_parse)

In [ ]:
rows = []

for _, r in df.iterrows():
    ann = r['constraint_annotation']

    # unwrap if needed
    if isinstance(ann, dict) and 'constraints' in ann:
        ann = ann['constraints']

    # guard: skip bad rows
    if not isinstance(ann, list):
        continue

    for c in ann:
        # guard again (THIS fixes your error)
        if not isinstance(c, dict):
            continue

        rows.append({
            'type': c.get('type'),
            'constraint': c.get('constraint'),
            'formal': c.get('formal'),
            'variables': c.get('variables'),
            'arity': c.get('arity')
        })

flat_df = pd.DataFrame(rows)

In [ ]:
flat_df['type'].value_counts()

In [ ]:
flat_df

In [ ]:
other_df = flat_df[flat_df['type'] == 'other']

In [ ]:
other_df[['constraint', 'formal', 'variables', 'arity']].head(30)

**We** analyzed the distribution and structure of constraint annotations in the dataset by flattening all extracted constraints into a unified representation. While canonical constraint types (e.g., counting, grouping, conditional, ordering) cover a substantial portion of cases, we observe a non-trivial “other” category that captures higher-order or structurally complex constraints that do not decompose cleanly into the predefined taxonomy. These include global or system-wide restrictions (e.g., “No two different tasks share the same hour, except …”), multi-variable attribute bindings (e.g., “Each piece’s final color must be brown, green, tan, …”), and structural or temporal constraints (e.g., “No pair of players can be teammates in two consecutive rounds” or “Each century’s paintings form one contiguous block”). Additionally, graph-structured constraints such as directed edge specifications (e.g., “There is a one-way street from X to Z”) are represented as individual relations but collectively induce a global structure. Overall, this suggests that while canonical types capture local relational reasoning, the “other” category systematically encodes higher-order global constraints typical of GRE-style logic puzzles, which are essential for full problem reconstruction but not easily reducible to pairwise or positional forms.


# Arity Distribution

In [ ]:
flat_df['is_other'] = flat_df['type'].eq('other')

In [ ]:
flat_df.groupby('is_other')['arity'].describe()

In [ ]:
flat_df.groupby('is_other')['arity'].mean()

In [ ]:
import matplotlib.pyplot as plt

flat_df.boxplot(column='arity', by='is_other')
plt.title("Arity distribution: other vs non-other")
plt.suptitle("")
plt.xlabel("is_other")
plt.ylabel("arity")
plt.show()

# Highest Arity Samples

In [ ]:
flat_df.sort_values('arity', ascending=False).head(1)

In [ ]:
flat_df.sort_values('arity', ascending=False).head(20)

We analyze the distribution of constraint arities in the extracted logic puzzle dataset. We observe that the highest-arity constraints correspond primarily to global or encoding-style constraints rather than inherently complex relational reasoning. For example, some constraints involve full-sequence positional mappings such as “All occurrences of the same digit stand for the same position mapping,” which operates over all positional variables (arity = 26). Similarly, frequency-based constraints such as “The letter A appears exactly two times in the sequence” and “Exactly three books are clothbound” apply over large subsets or the entire variable set (arity ≈ 12–18). In contrast, local relational constraints such as “Iris cannot immediately precede Salem” or “F and M perform simultaneously” involve only two variables (arity = 2).

This pattern indicates that arity primarily reflects the *scope* of a constraint rather than its logical complexity. High-arity constraints arise from system-wide encoding or counting requirements over full assignments, whereas low-arity constraints capture pairwise relations such as ordering, adjacency, or grouping. Therefore, arity alone is not a reliable indicator of reasoning difficulty; instead, it reflects how broadly a constraint is applied across the variable space.


# Constraint Category Distribution

In [ ]:
type_summary = (
    flat_df['type']
    .value_counts()
    .rename("count")
    .to_frame()
)

type_summary["percentage"] = (
    type_summary["count"] / type_summary["count"].sum() * 100
)

type_summary

In [ ]:
type_summary.sort_values("percentage", ascending=False)

In [ ]:
type_summary.round(2)

We observe that approximately 17% of extracted constraints are categorized as “other.” Manual inspection indicates that these constraints are not noise, but instead correspond to structurally global or encoding-style rules that are not easily reducible to the predefined canonical schema. Examples include full-domain positional mappings (e.g., digit-to-position consistency constraints over all slots), global cardinality constraints over sequences, and structural constraints such as graph or block-based conditions. While canonical categories capture local relational patterns (e.g., ordering, adjacency, grouping), the “other” category systematically encodes constraints that operate over the entire variable space or multiple interacting substructures. This suggests that the current taxonomy is sufficient for capturing the dominant constraint classes in GRE-style logic puzzles, while “other” serves as a meaningful residual class for higher-order global constraints.

# Question Type Analaysis

We categorize questions in the GRE-style logic puzzle dataset into seven functional types that reflect different reasoning operators applied over a fixed constraint system. Valid configuration questions evaluate satisfiability by asking whether a partial assignment or option is consistent with the given constraints, while complete solution questions require reconstructing the entire assignment of variables. must_be_true and must_be_false questions correspond to entailment and contradiction checking over the solution space, respectively. must_be_true evaluates whether a statement holds across all valid solutions, whereas must_be_false determines whether a statement holds in none of the valid solutions. Additionally, conditional_inference questions introduce hypothetical assumptions and require propagating their consequences through the constraint system, whereas optimization questions involve selecting extremal solutions under an implicit objective (e.g., earliest, latest, or maximum/minimum assignments). The residual other category captures atypical or hybrid question forms that do not conform cleanly to these canonical reasoning operators.

In [ ]:
question_types.head()

In [ ]:
question_types['question_type'].value_counts()

In [ ]:
qt_dist = question_types['question_type'].value_counts(normalize=True) * 100
qt_dist

In [ ]:
qt_summary = (
    question_types['question_type']
    .value_counts()
    .to_frame("count")
)

qt_summary["percentage"] = qt_summary["count"] / qt_summary["count"].sum() * 100

qt_summary.sort_values("percentage", ascending=False)

In [ ]:
question_types.groupby(['Puzzle_Number', 'question_type']).size()

# Question Distribution per puzzle (how mixed question intents are)

do puzzles have multiple question types?
or mostly single-intent questions per puzzle?

In [ ]:
question_types.groupby('Puzzle_Number')['question_type'].nunique().value_counts()

# Constraint-Question Interaction Matrix

In [ ]:
rows = []

for _, r in df.iterrows():
    puzzle_id = r['Puzzle_Number']   # ✅ carry it

    ann = r['constraint_annotation']

    if isinstance(ann, dict) and 'constraints' in ann:
        ann = ann['constraints']

    for c in ann:
        if not isinstance(c, dict):
            continue

        rows.append({
            'Puzzle_Number': puzzle_id,   # ✅ ADD THIS
            'type': c.get('type'),
            'constraint': c.get('constraint'),
            'formal': c.get('formal'),
            'variables': c.get('variables'),
            'arity': c.get('arity')
        })

flat_df = pd.DataFrame(rows)

In [ ]:
merged = flat_df.merge(
    question_types[['Puzzle_Number', 'question_type']],
    on='Puzzle_Number',
    how='inner'
)

In [ ]:
matrix = pd.crosstab(
    merged['type'],
    merged['question_type']
)

matrix

⚠️ Important conceptual point

This merge is valid because:

each puzzle has multiple constraints AND multiple questions

So implicitly we measure :  

constraint × question-type co-occurrence at puzzle level

NOT direct one-to-one mapping.

Good question — this is the key conceptual point behind your heatmap.

You already have the correct matrix, so now it’s just about interpretation.

---

# 📌 1. What your matrix actually is

You computed:

```python
matrix_norm = matrix.div(matrix.sum(axis=1), axis=0)
```

So every row sums to 1.

That means:

> Each row answers:
> **“Given a constraint type, how are its occurrences distributed across question types?”**

---

# 🧠 2. Row-normalized (what YOU have)

### Interpretation:

For each constraint type (row):

> “If I pick a constraint of this type, what is the probability it appears in each question type?”

---

### Example (your table)

Row: `grouping`

```
0.60 conditional_inference
0.18 valid_configuration
0.10 must_be_true
...
```

👉 Meaning:

> 60% of grouping constraints appear in conditional_inference questions

---

### So row-normalized =

✔ constraint → question distribution
✔ answers: “where do constraints go?”

---

# 📌 3. Column-normalized (the other version)

This is:

```python
matrix.div(matrix.sum(axis=0), axis=1)
```

### Interpretation:

For each question type (column):

> “Given a question type, what fraction of its constraints come from each constraint type?”

---

### Example (hypothetical)

Column: `conditional_inference`

```
0.30 grouping
0.20 counting
0.15 conditional
...
```

👉 Meaning:

> conditional_inference questions tend to involve grouping constraints 30% of the time

---

### So column-normalized =

✔ question → constraint distribution
✔ answers: “what builds this question type?”

---

# 🔥 4. Simple intuition difference

| Normalization     | Question being answered                        | Direction             |
| ----------------- | ---------------------------------------------- | --------------------- |
| Row-normalized    | What questions does this constraint appear in? | constraint → question |
| Column-normalized | What constraints appear in this question?      | question → constraint |

---

# 🧠 5. Why your current matrix looks “reasonable”

Your rows like:

```
grouping → 0.60 conditional_inference
```

mean:

> grouping constraints are heavily used in conditional reasoning tasks

That is a **valid structural insight about dataset design**

---

# 📊 6. When to use which (important for paper)

### Use row-normalized (your current one) when:

✔ analyzing constraint behavior
✔ studying how constraints “manifest” in reasoning tasks
✔ describing dataset construction

---

### Use column-normalized when:

✔ analyzing question difficulty
✔ understanding reasoning demands per question type
✔ model evaluation / failure analysis

---

# 🧩 7. One-line summary (ACL-style)

You can literally write this:

> Row-normalized matrices characterize how constraint types distribute across question types, whereas column-normalized matrices characterize the constraint composition of each question type.

---

# ⚡ Key takeaway

You didn’t compute two different things in magnitude — you computed:

> the same joint distribution viewed from two directions

---

If you want next step, I can help you make:
👉 a **single figure combining both views (dual heatmap or mirrored plot)** — this is often very strong in papers.


In [ ]:
matrix_norm = matrix.div(matrix.sum(axis=1), axis=0)

In [ ]:
matrix_norm

In [ ]:
matrix_norm = matrix_norm.fillna(0)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))

sns.heatmap(
    matrix_norm,
    annot=True,
    fmt=".2f",
    cmap="Blues",
    linewidths=0.5,
    cbar_kws={'label': 'Proportion'}
)

plt.title("Constraint Type × Question Type Interaction (Normalized)")
plt.xlabel("Question Type")
plt.ylabel("Constraint Type")
plt.tight_layout()
plt.show()

# Given a question type, what constraint types dominate?

In [ ]:
matrix_norm_col = matrix.div(matrix.sum(axis=0), axis=1).fillna(0)

In [ ]:
matrix_norm_col

In [ ]:
plt.figure(figsize=(10, 6))

sns.heatmap(
    matrix_norm_col,
    annot=True,
    fmt=".2f",
    cmap="Reds",
    linewidths=0.5,
    cbar_kws={'label': 'Proportion'}
)

plt.title("Constraint Type × Question Type (Column-Normalized)")
plt.xlabel("Question Type")
plt.ylabel("Constraint Type")
plt.tight_layout()
plt.show()

Good — now you can clearly see why these two matrices tell *different stories*.

Let’s make this precise and intuitive.

---

# 📌 1. What your column-normalized matrix means

You computed:

```python
matrix_norm_col = matrix.div(matrix.sum(axis=0), axis=1)
```

So each **column sums to 1**.

That means:

> For each **question type**, you are asking:
> “What proportion of its constraints come from each constraint type?”

---

# 🧠 2. Interpret your table correctly

Take this column:

## 🟦 `must_be_true`

You see values like:

* ordering → ~0.089
* grouping → ~0.074
* other → ~0.26 (very large)
* counting → ~0.178

### Meaning:

> Among all constraints that appear in *must_be_true* questions, ~26% are “other”-type constraints.

So:

✔ this is describing **question composition**

NOT constraint behavior.

---

# 📌 3. Compare with row-normalized (your earlier matrix)

Row-normalized asked:

> “Where does grouping appear?”

Column-normalized asks:

> “What builds must_be_true?”

---

# 🔥 4. Key conceptual difference (very important)

## Row-normalized (your first matrix)

[
P(\text{question type} \mid \text{constraint type})
]

Example:

> Given a grouping constraint → what question types use it?

---

## Column-normalized (this one)

[
P(\text{constraint type} \mid \text{question type})
]

Example:

> Given a must_be_true question → what constraints does it contain?

---

# 🧠 5. Why your “other” column looks strong

You observed:

* “other” dominates must_be_true (~0.26)
* also high in optimization (~0.43 in some rows)

### Interpretation:

This is actually important:

> The “other” constraint class is disproportionately used in **semantically heavy reasoning questions**

This suggests:

✔ your canonical taxonomy does NOT fully capture structural constraints in inference-heavy questions
✔ but is sufficient overall (as you already concluded)

---

# 📊 6. Simple intuition summary

| View              | Reads as              | Question answered                |
| ----------------- | --------------------- | -------------------------------- |
| Row-normalized    | constraint → question | “Where is this constraint used?” |
| Column-normalized | question → constraint | “What builds this question?”     |

---

# 📌 7. Why both are useful in your paper

You now have a clean duality:

### A. Structural view (row-normalized)

> How constraints distribute across reasoning tasks

### B. Compositional view (column-normalized)

> How reasoning tasks are constructed from constraint types

---

# 🧩 8. ACL-ready sentence you can actually use

> Row-normalized distributions capture how each constraint type is deployed across question types, whereas column-normalized distributions characterize the constraint composition of each question type, revealing systematic asymmetries between structural constraint usage and reasoning task formulation.

---

# ⚡ 9. One final intuition (important)

Think of it like:

* Row-normalized = “where does this ingredient go?”
* Column-normalized = “what is this dish made of?”

Same data — different direction of causality.

---

If you want next step, I can help you:
👉 turn this into a **two-panel ACL figure (left = row, right = column) with shared color semantics** — this is usually very strong visually.


In [ ]:
row_norm = matrix.div(matrix.sum(axis=1), axis=0).fillna(0)
col_norm = matrix.div(matrix.sum(axis=0), axis=1).fillna(0)

In [ ]:
vmin = min(row_norm.min().min(), col_norm.min().min())
vmax = max(row_norm.max().max(), col_norm.max().max())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(20, 8), sharey=True)

# -------------------------
# Left: Row-normalized
# -------------------------
sns.heatmap(
    row_norm,
    ax=axes[0],
    cmap="Blues",
    vmin=vmin,
    vmax=vmax,
    annot=True,          # 👈 show numbers
    fmt=".2f",           # 2 decimal places
    linewidths=0.5,
    cbar=True,
    cbar_kws={'label': 'Probability'}
)

axes[0].set_title("Row-normalized: P(Question Type | Constraint Type)")
axes[0].set_xlabel("Question Type")
axes[0].set_ylabel("Constraint Type")

# -------------------------
# Right: Column-normalized
# -------------------------
sns.heatmap(
    col_norm,
    ax=axes[1],
    cmap="Blues",
    vmin=vmin,
    vmax=vmax,
    annot=True,          # 👈 show numbers
    fmt=".1f",
    linewidths=0.5,
    cbar=True,
    cbar_kws={'label': 'Probability'}
)

axes[1].set_title("Column-normalized: P(Constraint Type | Question Type)")
axes[1].set_xlabel("Question Type")
axes[1].set_ylabel("")

plt.tight_layout()
plt.savefig("constraint_question_interaction.pdf", format="pdf", bbox_inches="tight")
plt.show()

The interaction between constraint structure and question formulation reveals a systematic asymmetry in how GRE-style logic puzzles are constructed. From the row-normalized perspective P(question type∣constraint type), most constraint categories exhibit a strong concentration toward conditional_inference questions, typically accounting for approximately 45–60% of occurrences across structural types such as grouping, counting, ordering, and conditional constraints. This indicates that constraint types are not uniformly distributed across reasoning tasks, but are instead predominantly instantiated in hypothetical or dependency-driven question forms. In contrast, question types such as valid_configuration and must_be_true show more moderate and stable distributions across constraint categories, suggesting that feasibility and entailment-based queries rely on a more heterogeneous mixture of structural constraints.

From the column-normalized perspective P(constraint type∣question type), we observe that question types differ substantially in their constraint composition. Conditional_inference questions are characterized by a relatively balanced mixture of constraint types, with notable contributions from counting and grouping constraints, reflecting their reliance on multi-step propagation across heterogeneous structural relations. In contrast, must_be_true and must_be_false questions show a higher relative concentration of the “other” constraint category, indicating that these questions frequently involve composite or non-canonical constraint forms that are not fully captured by the predefined taxonomy. Additionally, optimization and complete solution questions exhibit broader constraint diversity, consistent with their requirement to integrate multiple structural constraints simultaneously. Overall, these results suggest that while constraint types exhibit structured deployment patterns across reasoning tasks, question types induce a more variable and compositional mixture of constraint structures.

Figure X: Interaction between constraint types and question types in GRE-style logic puzzles. The left panel shows the distribution of question types conditioned on constraint type (row-normalized), while the right panel shows the distribution of constraint types conditioned on question type (column-normalized). Values indicate empirical probabilities. The shared color scale enables direct comparison across conditional directions, revealing asymmetries in how structural constraints are deployed versus how reasoning tasks are formulated.